# Классификация состояния здоровья студентов

**Тема:** Многоклассовая классификация с несбалансированными классами  
**Инструменты:** `pandas`, `RandomForestClassifier`

Предсказываем общее состояние здоровья студента (`health_condition`) по образу жизни и биометрии.

**Целевая переменная — 3 класса (сильный дисбаланс):**

| Класс | Доля |
|---|---|
| `at-risk` (в зоне риска) | ~86% |
| `unhealthy` (нездоров) | ~8% |
| `fit` (в форме) | ~6% |

Из-за сильного дисбаланса обычная accuracy обманчива (предсказав всем `at-risk`, уже получим ~86%). Поэтому используем **balanced accuracy** — среднюю полноту по классам.

---
## Шаги
1. Загрузка данных
2. Кодирование категориальных признаков
3. Обучение RandomForest
4. Оценка — balanced accuracy + CV + важность признаков
5. Генерация сабмита

## 1. Загрузка данных

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import balanced_accuracy_score, classification_report
from sklearn.model_selection import cross_val_score

train = pd.read_csv('/kaggle/input/student-health/train.csv')
test  = pd.read_csv('/kaggle/input/student-health/test.csv')

print('Размер train:', train.shape)
print('Размер test: ', test.shape)
train.head()

In [ ]:
# Распределение классов — виден сильный дисбаланс
counts = train['health_condition'].value_counts()
print(counts)
print()
print((counts / len(train) * 100).round(1).astype(str) + '%')

## 2. Кодирование категориальных признаков

Шесть колонок — текстовые категории. Большинство из них **порядковые** (есть естественный порядок: `low < medium < high`), поэтому переводим их в упорядоченные числа, а не в one-hot.

In [ ]:
ordinal_maps = {
    'diet_type':               {'veg': 1, 'non-veg': 0, 'balanced': 2},
    'stress_level':            {'low': 1, 'medium': 2, 'high': 3},
    'sleep_quality':           {'poor': 1, 'average': 2, 'good': 3},
    'physical_activity_level': {'sedentary': 1, 'moderate': 2, 'active': 3},
    'smoking_alcohol':         {'no': 0, 'occasional': 1, 'yes': 2},
    'gender':                  {'female': 1, 'male': 0, 'other': 2},
}
for col, mapping in ordinal_maps.items():
    train[col] = train[col].replace(mapping)
    test[col]  = test[col].replace(mapping)

# Заполняем пропуски медианой из train
train['sleep_duration'] = train['sleep_duration'].fillna(train['sleep_duration'].median())
test['sleep_duration']  = test['sleep_duration'].fillna(train['sleep_duration'].median())

features = ['sleep_duration', 'heart_rate', 'bmi', 'calorie_expenditure', 'step_count',
            'exercise_duration', 'water_intake', 'diet_type', 'stress_level',
            'sleep_quality', 'physical_activity_level', 'smoking_alcohol', 'gender']

X_train, y_train = train[features], train['health_condition']
X_test = test[features]
print('Признаков:', len(features))
print(X_train.shape)

## 3. Обучение модели

**RandomForestClassifier** с `max_depth=10` — крепкий честный бейзлайн.

> Замечание: более глубокие деревья или агрессивная генерация признаков (например, `stress_level * sleep_duration`) здесь **переобучаются** — этот чистый набор из 13 признаков обобщается лучше, чем усложнённые варианты.

In [ ]:
model = RandomForestClassifier(
    n_estimators=300,
    max_depth=10,
    min_samples_split=5,
    random_state=42,
    n_jobs=-1,
)
print('Обучение модели, подождите...')
model.fit(X_train, y_train)
print('Готово.')

## 4. Оценка качества

5-фолдовая кросс-валидация с метрикой **balanced accuracy** — той, что реально вознаграждает за правильное предсказание редких классов `fit` и `unhealthy`.

In [ ]:
print('Запуск кросс-валидации...')
cv_scores = cross_val_score(model, X_train, y_train, cv=5, scoring='balanced_accuracy', n_jobs=-1)
train_acc = balanced_accuracy_score(y_train, model.predict(X_train))

print('=' * 50)
print(' МЕТРИКИ (Balanced Accuracy):')
print(f'  • Train:                 {train_acc:.4f}')
print(f'  • Кросс-валидация (CV):  {cv_scores.mean():.4f} ± {cv_scores.std():.4f}')
print('=' * 50)

In [ ]:
# Метрики по каждому классу (важен recall на редких классах)
print(classification_report(y_train, model.predict(X_train)))

In [ ]:
# Важность признаков
importances = sorted(zip(features, model.feature_importances_), key=lambda x: x[1], reverse=True)

print(' ВАЖНОСТЬ ПРИЗНАКОВ (по убыванию):')
for name, imp in importances:
    print(f'  • {name:<24}: {imp:.2%}')

names, vals = zip(*importances)
plt.figure(figsize=(10, 4))
plt.bar(names, vals)
plt.title('Важность признаков')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

## 5. Генерация сабмита

In [ ]:
print('Генерация предсказаний для test...')
predictions = model.predict(X_test)

submission = pd.DataFrame({'id': test['id'], 'health_condition': predictions})
submission.to_csv('submission.csv', index=False)
print('submission.csv успешно сохранён!')
print(submission['health_condition'].value_counts())
submission.head()